In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage,HumanMessage
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain,create_history_aware_retriever
from langchain.prompts import MessagesPlaceholder

In [ ]:
GROQ_API_KEY = ""   #here enter your GROQ-API-KEY, Groq is a free open source platform for the models
# example          gak  = "gsk_KY6T1IsdPH58TTLA3WGdyb3FY2eeoMfo4eLf3396wdfn"
model = ChatGroq(groq_api_key = GROQ_API_KEY, model="llama3-70b-8192")
# currently we are using LLama3-70b-8192 , you can try other model also. And these models are not physically installed in your computer 

here i have gave GROQ_API_KEY directly, but it is a good practice to use .env file to store all api keys and credentials 

In [ ]:
my_embeddings = OllamaEmbeddings(model="gemma:2b")
data_base = FAISS.load_local("FAISS_DataBase" , my_embeddings,allow_dangerous_deserialization=True)
retriever = data_base.as_retriever()

In [ ]:
# You can change the system prompt , to change the behaviour of the model for the questions
system_prompt = (
    "Answer the question with respect to the context given, if the answer is not in the context then reply with the message that 'sorry, the question you asking is irrevalant or we don't have the answer'."
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history,if needed reformulate the question and answer it"
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , system_prompt),
        MessagesPlaceholder("chat history"),
        ("human" , "{input}")
    ]
)

In [21]:
history_aware_retriever = create_history_aware_retriever(model,retriever,prompt)
question_answer_chain=create_stuff_documents_chain(model,prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [23]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat history",
    output_messages_key="answer",
)

In [ ]:
def ask_question(question : str , chat_id : str) -> str:
    response = conversational_rag_chain.invoke(
        { "input" : question },
        config  = {
            "configurable": {"session_id": chat_id}    
        }
    )
    return response["answer"]

In [ ]:
#TO delete the Chat History run this cell
store = {}

In [ ]:
# here chat_id is a unqinue string for every chat History conversion, 
# if you want a new conversion give a new string for chat_id , 
# if you want to resume your chat, give the same chat Id 
ask_question("where is london?" , chat_id="chat1")